In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
%cd drive/MyDrive/

In [ ]:
!rm -rf FPL_forecast
!git clone https://github.com/bragehs/FPL_forecast.git

In [ ]:
%cd FPL_forecast/predictor/

In [1]:
file_path = '/content/drive/MyDrive/colab_fpl'
file_path

'/content/drive/MyDrive/colab_fpl'

In [1]:
import os
import torch
from training import train_model, hyperparameter_tuning
from model import FPLSequenceModel

In [2]:
file_path = os.getcwd() + "/processed_data"
file_path

'/Users/bragehs/Documents/FPL_forecast/predictor/processed_data'

In [3]:
X_train_numeric = torch.load(file_path + "/X_train.pt", weights_only=True)
X_train_static = torch.load(file_path + "/X_static_train.pt", weights_only=True)
y_train = torch.load(file_path + "/y_train.pt", weights_only=True)
X_val_numeric = torch.load(file_path + "/X_val.pt", weights_only=True)
X_val_static = torch.load(file_path + "/X_static_val.pt", weights_only=True)
y_val = torch.load(file_path + "/y_val.pt", weights_only=True)

print(f"Train sequences: {X_train_numeric.shape}, {X_train_static.shape}, Targets: {y_train.shape}")

Train sequences: torch.Size([162512, 5, 16]), torch.Size([162512, 14]), Targets: torch.Size([162512, 1])


In [4]:
pos_ids_train = torch.load(file_path + "/pos_train.pt", weights_only=True)
pos_ids_val = torch.load(file_path + "/pos_val.pt", weights_only=True)
fix_diff_train = torch.load(file_path + "/fixdiff_train.pt", weights_only=True)
fix_diff_val = torch.load(file_path + "/fixdiff_val.pt", weights_only=True)

In [5]:
print(f"Position IDs Train: {pos_ids_train.shape}, Val: {pos_ids_val.shape}")
print(f"Fixture Difficulty Train: {fix_diff_train.shape}, Val: {fix_diff_val.shape}")

Position IDs Train: torch.Size([162512, 5]), Val: torch.Size([29725, 5])
Fixture Difficulty Train: torch.Size([162512, 5]), Val: torch.Size([29725, 5])


In [6]:
print(fix_diff_train.unique())
print(pos_ids_train.unique())

tensor([0, 2, 3, 4, 5])
tensor([0, 1, 2, 3, 4])


In [ ]:
# Hyperparameter tuning
best_params = hyperparameter_tuning(X_train_numeric=X_train_numeric, y_train=y_train,X_train_static=X_train_static, 
                                    X_val_numeric=X_val_numeric, X_val_static=X_val_static, y_val=y_val,
                                    pos_ids_train=pos_ids_train, pos_ids_val=pos_ids_val,
                                    position_vocab_size= 6, position_embed_dim=16,
                                    fixdiff_ids_train=fix_diff_train, fixdiff_ids_val=fix_diff_val,
                                    fixture_diff_vocab_size=6, fixture_diff_embed_dim=16,
                                    epochs=1, n_trials=1, transform=False, num_workers=4,
                                    )

Running random search with 1 trials...

Trial 1/1
Params: {'learning_rate': 0.01, 'hidden_dim': 192, 'weight_decay': 0.01, 'lstm_layers': 2, 'dropout': 0.1, 'batch_size': 128}

Top 5 hyperparameter combinations:
1. RMSE: inf, Params: {'learning_rate': 0.01, 'hidden_dim': 192, 'weight_decay': 0.01, 'lstm_layers': 2, 'dropout': 0.1, 'batch_size': 128, 'rmse': inf}

Best hyperparameters: None
Best RMSE: inf

Training final model with best hyperparameters...


TypeError: 'NoneType' object is not subscriptable

In [8]:
print("\nTraining final model with best hyperparameters...")
model = FPLSequenceModel(
            numeric_seq_dim=X_train_numeric.shape[-1],
            static_dim=X_train_static.shape[-1],
            hidden_dim=128,
            lstm_layers=2,
            dropout=0.1,
            position_vocab_size=6,
            position_embed_dim=16,
            fixture_diff_vocab_size=6,
            fixture_diff_embed_dim=16,
        )


Training final model with best hyperparameters...


In [9]:
# Full training with best hyperparameters
train_model(
    model=model,
    X_train_numeric=X_train_numeric, X_train_static=X_train_static, y_train=y_train,
    X_val_numeric=X_val_numeric, X_val_static=X_val_static, y_val=y_val,
    pos_ids_train=pos_ids_train,
    pos_ids_val=pos_ids_val,
    fixdiff_ids_train=fix_diff_train,
    fixdiff_ids_val=fix_diff_val,
    learning_rate=0.01,
    weight_decay=0.0001,
    batch_size=32,
    epochs=100,
    verbose=2,
    transform=False,
    num_workers=4,
        )

Epoch  1/100: 100%|██████████| 5079/5079 [03:19<00:00, 25.44it/s, loss=7.3750, lr=0.01] 

Epoch 1 Training MSE: 4.7631


Epoch 1 validation RMSE: 1.9589
Epoch 1 validation MAE: 1.0504
Best model saved at epoch 1 with RMSE: 1.9589


Epoch  2/100: 100%|██████████| 5079/5079 [04:29<00:00, 18.83it/s, loss=0.9327, lr=0.00991] 

Epoch 2 Training MSE: 4.6643


Epoch 2 validation RMSE: 1.9662
Epoch 2 validation MAE: 0.9775


Epoch  3/100:   1%|          | 51/5079 [00:03<04:02, 20.74it/s, loss=0.6256, lr=0.00982] libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
libc++abi: terminating due to uncaught exception of type std::__1::system_error: Broken pipe
Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x10dda1bc0>
Traceback (most recent call last):
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/site-packages/torch/utils/data/dataloader.py", line 1568, in _shutdown_workers
    w.join(timeout=_utils.MP_STATUS_CHECK_INTERVAL)
  File "/opt/anaconda3/envs/fpl_helper/lib/python3.11/multiprocessing/process.py

KeyboardInterrupt: 